# Spotify Tracks — Limpeza e Geração do Dataset

**Grupo 4 — Insights Spotify**

Entrada: `dataset2(in).csv` (114.000 linhas × 20 colunas, faixas do Spotify com features de áudio).

Objetivo: diagnosticar, limpar e consolidar os dados brutos em um **dataset em nível de faixa**
(uma linha por música), pronto para análise.

Saídas em `data/processed/`:

| Arquivo | Grão | Uso |
|---|---|---|
| `spotify_tracks_limpo.csv` / `.parquet` | 1 linha por `track_id` | **Dataset principal** — análise por faixa |
| `spotify_tracks_genero_long.parquet` | 1 linha por `track_id` × gênero | Auxiliar — análise por gênero |
| `dicionario_dados.csv` | — | Dicionário das colunas do dataset principal |

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Resolve a raiz do repositorio a partir de onde o notebook estiver rodando
RAIZ = Path.cwd()
while not (RAIZ / "dataset2(in).csv").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

ARQUIVO_BRUTO = RAIZ / "dataset2(in).csv"
DIR_SAIDA = RAIZ / "data" / "processed"
DIR_SAIDA.mkdir(parents=True, exist_ok=True)

print("raiz  :", RAIZ)
print("bruto :", ARQUIVO_BRUTO, f"({ARQUIVO_BRUTO.stat().st_size / 1e6:.1f} MB)")
print("saida :", DIR_SAIDA)

raiz  : /home/msoares/projetos/insights-spotfy-grupo-4
bruto : /home/msoares/projetos/insights-spotfy-grupo-4/dataset2(in).csv (20.2 MB)
saida : /home/msoares/projetos/insights-spotfy-grupo-4/data/processed


## 1. Carga dos dados brutos

A primeira coluna do CSV é um índice sem nome herdado da exportação original — descartada em `index_col=0`.

In [ ]:
bruto = pd.read_csv(ARQUIVO_BRUTO, index_col=0)
bruto.index.name = "linha_original"

print(f"{bruto.shape[0]:,} linhas x {bruto.shape[1]} colunas".replace(",", "."))
bruto.head()

114.000 linhas x 20 colunas


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
linha_original,,,,,,,,,,,,,,,,,,,,
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [ ]:
bruto.dtypes.to_frame("dtype").assign(
    nao_nulos=bruto.notna().sum(),
    nulos=bruto.isna().sum(),
    unicos=bruto.nunique(),
)

,dtype,nao_nulos,nulos,unicos
track_id,str,114000,0,89741
artists,str,113999,1,31437
album_name,str,113999,1,46579
track_name,str,113999,1,73602
popularity,int64,114000,0,101
duration_ms,int64,114000,0,50697
explicit,bool,114000,0,2
danceability,float64,114000,0,1174
energy,float64,114000,0,2083
key,int64,114000,0,12


## 2. Diagnóstico

Antes de tocar em qualquer coisa, medimos o que está errado. Cada problema encontrado aqui
vira uma etapa explícita na seção 3.

In [ ]:
# 2.1 Registros sem identificacao
sem_identificacao = bruto[["artists", "album_name", "track_name"]].isna().any(axis=1)
print("Linhas sem artista/album/nome:", sem_identificacao.sum())
display(bruto[sem_identificacao])

Linhas sem artista/album/nome: 1


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
linha_original,,,,,,,,,,,,,,,,,,,,
65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,0.583,7,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


In [ ]:
# 2.2 Duplicatas
dup_exatas = bruto.duplicated(keep=False)
dup_id = bruto.duplicated("track_id", keep=False)

print(f"track_id distintos          : {bruto['track_id'].nunique():,}".replace(",", "."))
print(f"Linhas com track_id repetido: {dup_id.sum():,}".replace(",", "."))
print(f"Linhas 100% duplicadas      : {dup_exatas.sum():,}".replace(",", "."))

track_id distintos          : 89.741
Linhas com track_id repetido: 40.900
Linhas 100% duplicadas      : 894


In [ ]:
# Por que o mesmo track_id se repete? Comparamos as colunas entre linhas do mesmo track_id.
repetidos = bruto[dup_id]
variacao = repetidos.groupby("track_id").nunique().max()

print("Valores distintos por track_id (maximo observado):")
print(variacao[variacao > 1].to_string())
print()
print("Generos por faixa repetida:")
print(repetidos.groupby("track_id")["track_genre"].nunique().value_counts().sort_index().to_string())

Valores distintos por track_id (maximo observado):
popularity     2
track_genre    9

Generos por faixa repetida:
track_genre
1      342
2    11424
3     2955
4     1361
5      431
6      104
7       21
8        2
9        1


> **Conclusão:** a repetição de `track_id` **não é erro de carga** — a mesma faixa é catalogada em
> até 9 gêneros diferentes. Todas as colunas descritivas e de áudio são idênticas entre as cópias;
> só `popularity` varia (dois snapshots de coleta) e, em 342 casos, o gênero também se repete —
> essas sim são duplicatas verdadeiras.

In [ ]:
# 2.3 Valores invalidos nas colunas numericas
invalidos = pd.DataFrame({
    "regra": [
        "duration_ms == 0",
        "duration_ms < 30s",
        "tempo == 0 (BPM nao detectado)",
        "time_signature in (0, 1)",
        "popularity == 0",
    ],
    "linhas": [
        (bruto["duration_ms"] == 0).sum(),
        (bruto["duration_ms"] < 30_000).sum(),
        (bruto["tempo"] == 0).sum(),
        bruto["time_signature"].isin([0, 1]).sum(),
        (bruto["popularity"] == 0).sum(),
    ],
})
invalidos["% do total"] = (invalidos["linhas"] / len(bruto) * 100).round(2)
invalidos

,regra,linhas,% do total
0,duration_ms == 0,1,0.00
1,duration_ms < 30s,17,0.01
2,tempo == 0 (BPM nao detectado),157,0.14
3,"time_signature in (0, 1)",1136,1.00
4,popularity == 0,16020,14.05


In [ ]:
# 2.4 Distribuicoes das features numericas
bruto.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
popularity,114000.0,33.239,22.305,0.000,17.000,35.000,50.000,100.000
duration_ms,114000.0,228029.153,107297.713,0.000,174066.000,212906.000,261506.000,5237295.000
danceability,114000.0,0.567,0.174,0.000,0.456,0.580,0.695,0.985
energy,114000.0,0.641,0.252,0.000,0.472,0.685,0.854,1.000
key,114000.0,5.309,3.560,0.000,2.000,5.000,8.000,11.000
loudness,114000.0,-8.259,5.029,-49.531,-10.013,-7.004,-5.003,4.532
mode,114000.0,0.638,0.481,0.000,0.000,1.000,1.000,1.000
speechiness,114000.0,0.085,0.106,0.000,0.036,0.049,0.084,0.965
acousticness,114000.0,0.315,0.333,0.000,0.017,0.169,0.598,0.996
instrumentalness,114000.0,0.156,0.310,0.000,0.000,0.000,0.049,1.000


Leitura do diagnóstico:

- `tempo == 0` e `time_signature ∈ {0, 1}` são **falhas do algoritmo de análise do Spotify**, não zeros
  reais — viram `NaN` (`time_signature` válido é 3–7).
- `popularity == 0` é um valor **legítimo** (faixa sem streams no snapshot) — preservado.
- As features de áudio (`danceability`, `energy`, `valence`, …) já vêm normalizadas em [0, 1] e
  não apresentam valores fora de faixa.

## 3. Limpeza

Cada etapa registra quantas linhas entraram e quantas saíram, para o log de auditoria no final.

In [ ]:
log = []


# Formata inteiro com ponto como separador de milhar
def fmt(n: int) -> str:
    return f"{n:,}".replace(",", ".")


def registrar(etapa: str, antes: int, depois: int) -> None:
    log.append({"etapa": etapa, "antes": antes, "depois": depois, "removidas": antes - depois})
    print(f"{etapa}: {fmt(antes)} -> {fmt(depois)}  ({fmt(antes - depois)} removidas)")


df = bruto.copy()
n0 = len(df)

In [ ]:
# 3.1 Remover registros sem identificacao (1 linha: sem artista, album, nome e com duracao 0)
antes = len(df)
df = df[df[["artists", "album_name", "track_name"]].notna().all(axis=1)]
registrar("Sem identificacao", antes, len(df))

Sem identificacao: 114.000 -> 113.999  (1 removidas)


In [ ]:
# 3.2 Remover duplicatas exatas (mesma faixa, mesmo genero, tudo igual)
antes = len(df)
df = df.drop_duplicates()
registrar("Duplicatas exatas", antes, len(df))

Duplicatas exatas: 113.999 -> 113.549  (450 removidas)


In [ ]:
# 3.3 Sobras de (track_id, track_genre) repetidos: mantem a linha de maior popularidade
antes = len(df)
df = (
    df.sort_values("popularity", ascending=False)
      .drop_duplicates(subset=["track_id", "track_genre"], keep="first")
      .sort_index()
)
registrar("(track_id, genero) repetidos", antes, len(df))

(track_id, genero) repetidos: 113.549 -> 113.549  (0 removidas)


In [ ]:
# 3.4 Valores sentinela -> NaN
df = df.copy()
df.loc[df["tempo"] == 0, "tempo"] = np.nan
df.loc[~df["time_signature"].between(3, 7), "time_signature"] = np.nan
df.loc[df["duration_ms"] == 0, "duration_ms"] = np.nan

print("NaN introduzidos:")
print(df[["tempo", "time_signature", "duration_ms"]].isna().sum().to_string())

NaN introduzidos:
tempo              157
time_signature    1130
duration_ms          0


### 3.5 Consolidação para nível de faixa

Aqui o dataset muda de grão: de *faixa × gênero* para **uma linha por faixa**. Os gêneros são
agregados em `generos` (lista separada por `;`) e contados em `n_generos`; `popularity` fica com o
maior valor entre os snapshots. Antes disso, salvamos a versão longa — ela é a base certa para
qualquer análise por gênero.

In [ ]:
# Versao longa (faixa x genero), ja limpa - insumo para analises por genero
df_long = df.reset_index(drop=True)
print(f"Versao longa: {len(df_long):,} linhas".replace(",", "."))

COLS_FAIXA = [
    "track_id", "artists", "album_name", "track_name", "duration_ms", "explicit",
    "danceability", "energy", "key", "loudness", "mode", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo", "time_signature",
]

antes = len(df_long)
faixas = (
    df_long.groupby("track_id", as_index=False)
    .agg(
        **{col: (col, "first") for col in COLS_FAIXA if col != "track_id"},
        popularity=("popularity", "max"),
        generos=("track_genre", lambda s: ";".join(sorted(set(s)))),
        n_generos=("track_genre", "nunique"),
    )
)
registrar("Consolidacao por faixa", antes, len(faixas))

Versao longa: 113.549 linhas


Consolidacao por faixa: 113.549 -> 89.740  (23.809 removidas)


In [ ]:
# Genero principal: primeiro em ordem alfabetica entre os gêneros da faixa.
# Criterio arbitrario e assumido - os dados nao trazem hierarquia ou peso entre generos.
faixas["genero_principal"] = faixas["generos"].str.split(";").str[0]

faixas["n_generos"].value_counts().sort_index().to_frame("faixas")

,faixas
n_generos,
1,73441
2,11424
3,2955
4,1361
5,431
6,104
7,21
8,2
9,1


## 4. Colunas derivadas

Traduções de códigos numéricos e agrupamentos que evitam repetir a mesma transformação em cada análise.

In [ ]:
NOTAS = ["C", "C#/Db", "D", "D#/Eb", "E", "F", "F#/Gb", "G", "G#/Ab", "A", "A#/Bb", "B"]

faixas["duracao_min"] = (faixas["duration_ms"] / 60_000).round(3)

faixas["duracao_categoria"] = pd.cut(
    faixas["duracao_min"],
    bins=[0, 2.5, 5, 10, np.inf],
    labels=["Curta (<2.5min)", "Media (2.5-5min)", "Longa (5-10min)", "Muito longa (>10min)"],
    right=False,
)

faixas["popularidade_faixa"] = pd.cut(
    faixas["popularity"],
    bins=[-1, 0, 25, 50, 75, 100],
    labels=["Sem streams", "Baixa (1-25)", "Media (26-50)", "Alta (51-75)", "Muito alta (76-100)"],
)

faixas["tonalidade"] = faixas["key"].map(dict(enumerate(NOTAS)))
faixas["modo"] = faixas["mode"].map({0: "Menor", 1: "Maior"})
faixas["tonalidade_completa"] = faixas["tonalidade"] + " " + faixas["modo"]

faixas["e_instrumental"] = faixas["instrumentalness"] > 0.5
faixas["e_ao_vivo"] = faixas["liveness"] > 0.8
faixas["n_artistas"] = faixas["artists"].str.count(";") + 1
faixas["artista_principal"] = faixas["artists"].str.split(";").str[0]

faixas.head(3)

,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,popularity,generos,n_generos,genero_principal,duracao_min,duracao_categoria,popularidade_faixa,tonalidade,modo,tonalidade_completa,e_instrumental,e_ao_vivo,n_artistas,artista_principal
0,0000vdREvCVMxbQTkS888c,Rill,Lolly,Lolly,160725.0,True,0.910,0.374,8,-9.844,0,0.1990,0.07570,0.00301,0.154,0.432,104.042,4.0,44,german,1,german,2.679,Media (2.5-5min),Media (26-50),G#/Ab,Menor,G#/Ab Menor,False,False,1,Rill
1,000CC8EParg64OmTxVnZ0p,Glee Cast,Glee Love Songs,It's All Coming Back To Me Now (Glee Cast Vers...,322933.0,False,0.269,0.516,0,-7.361,1,0.0366,0.40600,0.00000,0.117,0.341,178.174,4.0,47,club,1,club,5.382,Longa (5-10min),Media (26-50),C,Maior,C Maior,False,False,1,Glee Cast
2,000Iz0K615UepwSJ5z2RE5,Paul Kalkbrenner;Pig&Dan,X,Böxig Leise - Pig & Dan Remix,515360.0,False,0.686,0.560,5,-13.264,0,0.0462,0.00114,0.18100,0.111,0.108,119.997,4.0,22,minimal-techno,1,minimal-techno,8.589,Longa (5-10min),Baixa (1-25),F,Menor,F Menor,False,False,2,Paul Kalkbrenner


## 5. Validação

Testes que precisam passar antes da exportação. Se algum falhar, o notebook para aqui.

In [ ]:
assert faixas["track_id"].is_unique, "track_id deveria ser unico apos a consolidacao"
assert faixas[["track_id", "artists", "album_name", "track_name"]].notna().all().all(), "chaves com nulo"
assert faixas["popularity"].between(0, 100).all(), "popularity fora de 0-100"
assert faixas["n_generos"].ge(1).all(), "faixa sem genero"

FEATURES_01 = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence",
]
for col in FEATURES_01:
    assert faixas[col].between(0, 1).all(), f"{col} fora de [0, 1]"

assert faixas["tempo"].dropna().gt(0).all(), "tempo <= 0 remanescente"
assert faixas["time_signature"].dropna().between(3, 7).all(), "time_signature invalido remanescente"

print("Todas as validacoes passaram.")

Todas as validacoes passaram.


In [ ]:
# Nulos residuais - esperados apenas nas colunas onde sentinelas viraram NaN
residuais = faixas.isna().sum()
residuais = residuais[residuais > 0].to_frame("nulos")
residuais["%"] = (residuais["nulos"] / len(faixas) * 100).round(2)
residuais

,nulos,%
tempo,157,0.17
time_signature,1008,1.12


In [ ]:
# Log de auditoria da limpeza
auditoria = pd.DataFrame(log)
print(f"Bruto: {n0:,} linhas -> Final: {len(faixas):,} faixas unicas".replace(",", "."))
auditoria

Bruto: 114.000 linhas -> Final: 89.740 faixas unicas


,etapa,antes,depois,removidas
0,Sem identificacao,114000,113999,1
1,Duplicatas exatas,113999,113549,450
2,"(track_id, genero) repetidos",113549,113549,0
3,Consolidacao por faixa,113549,89740,23809


## 6. Dicionário de dados

In [ ]:
DESCRICOES = {
    "track_id": "Identificador unico da faixa no Spotify (chave primaria)",
    "artists": "Artistas da faixa, separados por ';'",
    "artista_principal": "Primeiro artista creditado (derivada)",
    "n_artistas": "Quantidade de artistas creditados (derivada)",
    "album_name": "Album em que a faixa aparece",
    "track_name": "Nome da faixa",
    "popularity": "Popularidade 0-100 (maior valor entre os snapshots)",
    "popularidade_faixa": "Popularidade agrupada em categorias (derivada)",
    "duration_ms": "Duracao em milissegundos (NaN quando o bruto trazia 0)",
    "duracao_min": "Duracao em minutos (derivada)",
    "duracao_categoria": "Duracao agrupada em categorias (derivada)",
    "explicit": "Faixa com conteudo explicito",
    "danceability": "Adequacao para danca (0-1)",
    "energy": "Intensidade e atividade percebidas (0-1)",
    "key": "Tom da faixa em pitch class (0=C ... 11=B)",
    "tonalidade": "Tom por extenso (derivada de key)",
    "loudness": "Volume medio em dB (tipicamente -60 a 0)",
    "mode": "Modo: 0=menor, 1=maior",
    "modo": "Modo por extenso (derivada de mode)",
    "tonalidade_completa": "Tom + modo, ex.: 'C Maior' (derivada)",
    "speechiness": "Presenca de palavra falada (0-1)",
    "acousticness": "Confianca de que a faixa e acustica (0-1)",
    "instrumentalness": "Confianca de que nao ha vocais (0-1)",
    "e_instrumental": "instrumentalness > 0.5 (derivada)",
    "liveness": "Indicio de plateia na gravacao (0-1)",
    "e_ao_vivo": "liveness > 0.8 (derivada)",
    "valence": "Positividade musical (0-1)",
    "tempo": "Andamento em BPM (NaN quando nao detectado)",
    "time_signature": "Formula de compasso, 3 a 7 (NaN quando invalida)",
    "generos": "Generos da faixa, separados por ';'",
    "n_generos": "Quantidade de generos em que a faixa aparece (derivada)",
    "genero_principal": "Primeiro genero em ordem alfabetica (derivada, criterio arbitrario)",
}

dicionario = pd.DataFrame({
    "coluna": faixas.columns,
    "tipo": [str(t) for t in faixas.dtypes],
    "nulos": faixas.isna().sum().values,
    "descricao": [DESCRICOES.get(col, "") for col in faixas.columns],
})
assert (dicionario["descricao"] != "").all(), "coluna sem descricao no dicionario"
dicionario

,coluna,tipo,nulos,descricao
0,track_id,str,0,Identificador unico da faixa no Spotify (chave...
1,artists,str,0,"Artistas da faixa, separados por ';'"
2,album_name,str,0,Album em que a faixa aparece
3,track_name,str,0,Nome da faixa
4,duration_ms,float64,0,Duracao em milissegundos (NaN quando o bruto t...
5,explicit,bool,0,Faixa com conteudo explicito
6,danceability,float64,0,Adequacao para danca (0-1)
7,energy,float64,0,Intensidade e atividade percebidas (0-1)
8,key,int64,0,Tom da faixa em pitch class (0=C ... 11=B)
9,loudness,float64,0,Volume medio em dB (tipicamente -60 a 0)


## 7. Exportação

In [ ]:
COLUNAS_FINAIS = [
    # Identificacao
    "track_id", "track_name", "artists", "artista_principal", "n_artistas", "album_name",
    # Genero
    "generos", "genero_principal", "n_generos",
    # Popularidade
    "popularity", "popularidade_faixa",
    # Duracao
    "duration_ms", "duracao_min", "duracao_categoria",
    # Features de audio
    "explicit", "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo",
    # Estrutura musical
    "key", "tonalidade", "mode", "modo", "tonalidade_completa", "time_signature",
    # Flags
    "e_instrumental", "e_ao_vivo",
]
assert sorted(COLUNAS_FINAIS) == sorted(faixas.columns), "divergencia entre COLUNAS_FINAIS e o dataframe"

final = faixas[COLUNAS_FINAIS]
final.head()

,track_id,track_name,artists,artista_principal,n_artistas,album_name,generos,genero_principal,n_generos,popularity,popularidade_faixa,duration_ms,duracao_min,duracao_categoria,explicit,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,key,tonalidade,mode,modo,tonalidade_completa,time_signature,e_instrumental,e_ao_vivo
0,0000vdREvCVMxbQTkS888c,Lolly,Rill,Rill,1,Lolly,german,german,1,44,Media (26-50),160725.0,2.679,Media (2.5-5min),True,0.910,0.374,-9.844,0.1990,0.075700,0.00301,0.1540,0.432,104.042,8,G#/Ab,0,Menor,G#/Ab Menor,4.0,False,False
1,000CC8EParg64OmTxVnZ0p,It's All Coming Back To Me Now (Glee Cast Vers...,Glee Cast,Glee Cast,1,Glee Love Songs,club,club,1,47,Media (26-50),322933.0,5.382,Longa (5-10min),False,0.269,0.516,-7.361,0.0366,0.406000,0.00000,0.1170,0.341,178.174,0,C,1,Maior,C Maior,4.0,False,False
2,000Iz0K615UepwSJ5z2RE5,Böxig Leise - Pig & Dan Remix,Paul Kalkbrenner;Pig&Dan,Paul Kalkbrenner,2,X,minimal-techno,minimal-techno,1,22,Baixa (1-25),515360.0,8.589,Longa (5-10min),False,0.686,0.560,-13.264,0.0462,0.001140,0.18100,0.1110,0.108,119.997,5,F,0,Menor,F Menor,4.0,False,False
3,000RDCYioLteXcutOjeweY,Teeje Week,Jordan Sandhu,Jordan Sandhu,1,Teeje Week,hip-hop,hip-hop,1,62,Alta (51-75),190203.0,3.170,Media (2.5-5min),False,0.679,0.770,-3.537,0.1900,0.058300,0.00000,0.0825,0.839,161.721,0,C,1,Maior,C Maior,4.0,False,False
4,000qpdoc97IMTBvF8gwcpy,Tief,Paul Kalkbrenner,Paul Kalkbrenner,1,Zeit,minimal-techno,minimal-techno,1,19,Baixa (1-25),331240.0,5.521,Longa (5-10min),False,0.519,0.431,-13.606,0.0291,0.000964,0.72000,0.0916,0.234,129.971,6,F#/Gb,0,Menor,F#/Gb Menor,4.0,True,False


In [ ]:
saidas = {
    "spotify_tracks_limpo.csv": lambda p: final.to_csv(p, index=False),
    "spotify_tracks_limpo.parquet": lambda p: final.to_parquet(p, index=False),
    "spotify_tracks_genero_long.parquet": lambda p: df_long.to_parquet(p, index=False),
    "dicionario_dados.csv": lambda p: dicionario.to_csv(p, index=False),
    "log_limpeza.csv": lambda p: auditoria.to_csv(p, index=False),
}

for nome, escrever in saidas.items():
    caminho = DIR_SAIDA / nome
    escrever(caminho)
    print(f"{nome:38s} {caminho.stat().st_size / 1e6:7.2f} MB")

print()
print(f"Dataset principal: {len(final):,} faixas x {final.shape[1]} colunas".replace(",", "."))

spotify_tracks_limpo.csv                 24.07 MB
spotify_tracks_limpo.parquet              8.98 MB


spotify_tracks_genero_long.parquet        8.65 MB
dicionario_dados.csv                      0.00 MB
log_limpeza.csv                           0.00 MB

Dataset principal: 89.740 faixas x 32 colunas


In [ ]:
# Conferencia final: le de volta o parquet e compara
conferencia = pd.read_parquet(DIR_SAIDA / "spotify_tracks_limpo.parquet")
assert conferencia.shape == final.shape, "shape divergente na releitura"
assert conferencia["track_id"].is_unique, "track_id duplicado no arquivo exportado"
print("Releitura OK:", conferencia.shape)
conferencia.sample(5, random_state=42)

Releitura OK: (89740, 32)


,track_id,track_name,artists,artista_principal,n_artistas,album_name,generos,genero_principal,n_generos,popularity,popularidade_faixa,duration_ms,duracao_min,duracao_categoria,explicit,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,key,tonalidade,mode,modo,tonalidade_completa,time_signature,e_instrumental,e_ao_vivo
56975,4x6JPITCakOVB58LPv9GRA,Tag am Meer - Stereoact Remix,Troglauer;Stereoact,Troglauer,2,Tag am Meer (Stereoact Remix),party,party,1,33,Media (26-50),234375.0,3.906,Media (2.5-5min),False,0.728,0.866,-6.023,0.1740,0.0217,0.000000,0.2740,0.7810,128.019,5,F,1,Maior,F Maior,4.0,False,False
84350,7K2oNX8UnvtebYRQtvHuUZ,睡蓮花,Shonan No Kaze,Shonan No Kaze,1,睡蓮花,j-pop,j-pop,1,43,Media (26-50),435186.0,7.253,Longa (5-10min),False,0.483,0.986,-2.446,0.3630,0.1260,0.000000,0.0825,0.3590,156.147,1,C#/Db,1,Maior,C#/Db Maior,4.0,False,False
18425,1b5yadCeqXSlIg9QJuY6kA,Aşk Benim Neyime,Serkan Kaya,Serkan Kaya,1,Miras,show-tunes,show-tunes,1,20,Baixa (1-25),288518.0,4.809,Media (2.5-5min),False,0.630,0.868,-3.561,0.0517,0.1460,0.000035,0.1170,0.5660,119.779,5,F,0,Menor,F Menor,4.0,False,False
61780,5NJKTfyxjv4GlDL6kRQ5fg,Nazar Değmesin,Gülşen,Gülşen,1,Of Of,turkish,turkish,1,41,Media (26-50),278186.0,4.636,Media (2.5-5min),False,0.504,0.504,-6.046,0.0515,0.3240,0.000000,0.1330,0.4680,139.480,5,F,0,Menor,F Menor,4.0,False,False
85825,7eVDCIw2l8u3Fy13xJyfaD,Holy War,Hans Zimmer,Hans Zimmer,1,Dune (Original Motion Picture Soundtrack),german,german,1,46,Media (26-50),260532.0,4.342,Media (2.5-5min),False,0.167,0.125,-20.997,0.0351,0.7180,0.962000,0.0941,0.0285,95.804,7,G,1,Maior,G Maior,4.0,True,False
